## Compute stability using MPRelaxSet

The goal is to allow comparision with MP database. In the meantime, prototype the code for MP support layer.

In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [42]:
from ase.io import read
from aiida import orm
from monty.serialization import loadfn, dumpfn
import numpy as np
from aiida_vasp.workchains import VaspHybridBandsWorkChain
from aiida_vasp.workchains.v2 import VaspHybridBandUpdater, VaspRelaxUpdater
from aiida_grouppathx import GroupPathX, decorate_with_uuid
from tqdm import tqdm

from aiida_grouppathx.launch_manager import GroupLauncher

In [3]:
basepath = GroupPathX('hc-antiperovskites-mbj')
structpath = basepath['structures']

In [4]:
structpath.show_tree()


structures
├── Ba3GeO_cubic *
├── Ba3GeS_cubic *
├── Ba3GeSe_cubic *
├── Ba3GeTe_cubic *
├── Ba3PbO_cubic *
├── Ba3PbS_cubic *
├── Ba3PbSe_cubic *
├── Ba3PbTe_cubic *
├── Ba3SbO_cubic *
├── Ba3SbS_cubic *
├── Ba3SbSe_cubic *
├── Ba3SbTe_cubic *
├── Ba3SiO_cubic *
├── Ba3SiS_cubic *
├── Ba3SiSe_cubic *
├── Ba3SiTe_cubic *
├── Ba3SnO_cubic *
├── Ba3SnS_cubic *
├── Ba3SnSe_cubic *
├── Ba3SnTe_cubic *
├── Ca3GeO_cubic *
├── Ca3GeS_cubic *
├── Ca3GeSe_cubic *
├── Ca3GeTe_cubic *
├── Ca3PbO_cubic *
├── Ca3PbS_cubic *
├── Ca3PbSe_cubic *
├── Ca3PbTe_cubic *
├── Ca3SbO_cubic *
├── Ca3SbS_cubic *
├── Ca3SbSe_cubic *
├── Ca3SbTe_cubic *
├── Ca3SiO_cubic *
├── Ca3SiS_cubic *
├── Ca3SiSe_cubic *
├── Ca3SiTe_cubic *
├── Ca3SnO_cubic *
├── Ca3SnS_cubic *
├── Ca3SnSe_cubic *
├── Ca3SnTe_cubic *
├── Mg3GeO_cubic *
├── Mg3GeS_cubic *
├── Mg3GeSe_cubic *
├── Mg3GeTe_cubic *
├── Mg3PbO_cubic *
├── Mg3PbS_cubic *
├── Mg3PbSe_cubic *
├── Mg3PbTe_cubic *
├── Mg3SbO_cubic *
├── Mg3SbS_cubic *
├── Mg3SbSe_cub

In [5]:
def gammaoffset(kpts):
    offset = []
    for k in kpts:
        if k % 2 == 1:
            offset.append(0)
        else:
            offset.append(1/2)
    return offset

In [141]:
    from pymatgen.io.vasp.sets import MPRelaxSet


In [ ]:
MPRelaxSet

In [38]:
def callback(node, label):
    """Callback for preparing and launching jobs"""
    from pymatgen.io.vasp.sets import MPRelaxSet
    relax_set = MPRelaxSet(node.get_pymatgen())
    upd = VaspRelaxUpdater()
    overrides = {key.lower(): value for key, value in relax_set.incar.items()}
    overrides.pop('nsw')
    overrides.pop('isif')
    overrides['icharg'] = None
    overrides['ibrion'] = None
    upd.set_name = 'MPRelaxSet'
    upd.apply_preset(node, 
                     overrides=overrides,
                                    code='vasp-6.3.0@sugon-xian-v2', label=f'{label} {node.label} MPRelaxSet'
                                         )
    upd.set_resources(num_machines=1, tot_num_mpiprocs=8)
    upd.set_options(max_wallclock_seconds=3600 * 24, queue_name='xahcnormal')
    kmesh = relax_set.kpoints.kpts[0]
    upd.set_kpoints_mesh(kmesh, offset=gammaoffset(kmesh))
    # Updated version
    upd.set_relax_settings(algo='rd')
    upd.set_incar(symprec=1e-9)
    running = upd.submit()
    return running, label

## Analysis

In [138]:
import os
os.environ['https_proxy'] = 'http://localhost:7890'

In [114]:
from mp_api.client import MPRester
from pymatgen.entries.computed_entries import ComputedEntry, ComputedStructureEntry
from pymatgen.entries.compatibility import MaterialsProject2020Compatibility

In [128]:
rester = MPRester()

In [129]:
node.outputs.misc['total_energies']['energy_extrapolated']

-14.7841885

In [130]:
out_entries = []
for path in basepath['mp_relax'].fast_iter:
    node = path.get_node()
    entry = ComputedStructureEntry(node.outputs.relax.structure.get_pymatgen(), energy=node.outputs.misc['total_energies']['energy_extrapolated'])
    entry.parameters['software'] = 'vasp'
    entry.parameters['run_type'] = 'GGA+U'
    out_entries.append(entry)

In [131]:
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter

In [132]:
comp = MaterialsProject2020Compatibility(check_potcar=False)
comp.process_entries(out_entries);

/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/pymatgen/entries/compatibility.py:1039: UserWarning: Failed to guess oxidation states for Entry None (Mg3PbSe). Assigning anion correction to only the most electronegative atom.
  warnings.warn(
/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/pymatgen/entries/compatibility.py:1039: UserWarning: Failed to guess oxidation states for Entry None (Sr3PbS). Assigning anion correction to only the most electronegative atom.
  warnings.warn(
/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/pymatgen/entries/compatibility.py:1039: UserWarning: Failed to guess oxidation states for Entry None (Ba3GeTe). Assigning anion correction to only the most electronegative atom.
  warnings.warn(
/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/pymatgen/entries/compatibility.py:1039: UserWarning: Failed to guess oxidation states for Entry None (Mg3SbSe). Assigning anion correction to only the most electron

In [139]:
e_above_hulls = {}
for test_entry in out_entries:
    formula = test_entry.composition.reduced_formula
    print('Checking: ', test_entry.composition.reduced_formula)
    entries = rester.get_entries_in_chemsys([x.symbol for x in test_entry.composition.keys()])
    phd = PhaseDiagram(entries + [test_entry])
    e_above_hull = phd.get_e_above_hull(test_entry)
    print(f'Energy above hull: {e_above_hull:.3f}')
    e_above_hulls[formula] = e_above_hull

Checking:  Mg3PbSe


Retrieving ThermoDoc documents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 61/61 [00:00<00:00, 1541280.39it/s]


Energy above hull: 0.384
Checking:  Sr3SiSe


Retrieving ThermoDoc documents: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 83/83 [00:00<00:00, 821054.79it/s]


Energy above hull: 0.448
Checking:  Sr3PbS


...

Energy above hull: 0.155
Checking:  Sr3SiTe


Retrieving ThermoDoc documents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 3125943.55it/s]


Energy above hull: 0.681
Checking:  Sr3GeTe


Retrieving ThermoDoc documents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 54/54 [00:00<00:00, 1509949.44it/s]

Energy above hull: 0.627


In [140]:
e_above_hulls

{'Mg3PbSe': 0.3836007560000003,
 'Sr3SiSe': 0.4483974860000002,
 'Sr3PbS': 0.12477867400000076,
 'Ba3SiS': 0.34250982350000037,
 'Ba3GeTe': 0.6024177880000008,
 'Mg3SbSe': 0.5042458844906781,
 'Ca3GeTe': 0.6726682099999999,
 'Ca3SbO': 0.15599275599999984,
 'Mg3SbO': 0.2569410464906774,
 'Ca3SbSe': 0.4297654940000002,
 'Mg3GeTe': 0.9472102339999999,
 'Ba3SbSe': 0.3811783536666664,
 'Sr3SiO': 0.014787732500000317,
 'Ba3PbO': 0.0,
 'Ba3PbSe': 0.23816601999999998,
 'Ca3PbSe': 0.21107466800000063,
 'Mg3GeO': 0.18689170599999994,
 'Ca3GeO': 0.007172269999998981,
 'Ca3GeS': 0.29003737600000035,
 'Mg3GeS': 0.5250689879999992,
 'Sr3GeSe': 0.4074774255000002,
 'Ba3PbS': 0.14191607599999978,
 'Ca3SiTe': 0.7290936884999999,
 'Sr3SiS': 0.32453067800000124,
 'Ba3SiTe': 0.656220357500001,
 'Mg3SbS': 0.4016272184906775,
 'Ca3SbS': 0.3200982459999997,
 'Sr3TePb': 0.5089002959999993,
 'Ba3SiO': 0.04836418300000034,
 'Mg3SiTe': 1.0302665679999996,
 'Sr3PbO': 0.0,
 'Sr3SbTe': 0.6261792512222226,
 'Ba3SbTe